# 06 FAQ 품질 검증

05가 만든 라운드 파일을 검사해 학습용 FAQ를 내보낸다. **QA를 지우지 않고 검사 결과를 기록한 뒤, 내보낼 때만 거른다**(기준을 바꿔도 재생성 불필요).

| 단계 | 처리 | 모듈 |
|---|---|---|
| 결정적 검사(계산·한도·등급·종·숫자·조문·근거 표기·needs_review) | 버림 / 수정 / 표시 | `data/faq_checks.py` |
| LLM 심판(gemma4:12b) — 독해 오류·지어내기·질문 품질 | 버림 | `data/faq_judge.py` |
| 중복(같은 단위 묶음 안, 심판 통과 QA끼리) | 버림 | `data/faq_checks.py` |
| 단위당 상한 max(5, 원문 길이/40) — 바꿔 말한 질문이 많은 군집부터 깎음 | 버림(`unit_cap`으로 기록) | `data/faq_checks.py` |
| 단위별 잔여 QA 집계 — 0이 된 단위 보고 | 보고 | 이 노트북 |

심판을 중복보다 먼저 한다. 2026-10-13 첫 전량 실행은 중복 → 심판 순서여서, 대표 QA가 심판에 걸린 그룹 417개에서 심판을 받지 않은 중복본 721건까지 함께 버려졌다.

단위 구성은 05와 같은 `data/faq_units.py`를 쓴다(검사하려면 FAQ를 만든 원문·참조를 똑같이 복원해야 한다).

In [1]:
# ===== 1: 설정 =====
import sys, json, re, time, importlib
from pathlib import Path
from collections import Counter, defaultdict

PROJECT_DIR = Path.home() / "AI_programming" / "petmint-rag-chatbot"
sys.path.insert(0, str(PROJECT_DIR))
FAQ_DIR = PROJECT_DIR / "data" / "faq"
INPUT_DIR = FAQ_DIR                       # 05 라운드 파일 위치(시험할 때만 바꾼다)
OUTPUT_DIR = FAQ_DIR / "checked"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
SPECIES = ["강아지", "고양이"]

import data.rag_context as rag_context
import data.faq_units as faq_units
import data.faq_checks as faq_checks
import data.faq_judge as faq_judge
for m in (rag_context, faq_units, faq_checks, faq_judge):
    importlib.reload(m)

builder = rag_context.ContextBuilder()      # 05와 같은 청크·그래프·Milvus — 참조를 생성 때와 같게 복원
units = {sp: {faq_units.unit_id(u): u for u in faq_units.make_units(builder.chunks[sp])} for sp in SPECIES}
# 도입 문장만 있는 단위는 make_units가 뺀다 — 그 단위로 이미 생성된 QA를 셀 3에서 '제외 단위'로 표시하려고 따로 둔다
excluded_units = {sp: faq_units.intro_only_unit_ids(builder.chunks[sp]) for sp in SPECIES}
print({sp: len(v) for sp, v in units.items()}, "단위 | 제외(도입 문장)", {sp: len(v) for sp, v in excluded_units.items()})

{'강아지': 301, '고양이': 274} 단위 | 제외(도입 문장) {'강아지': 1, '고양이': 1}


In [2]:
# ===== 2: 05 라운드 파일 읽기 =====
def load_rounds(species: str) -> list[dict]:
    """끝난 생성 라운드만 읽는다. 05가 같은 폴더에 쓰는 ..._roundN_failures.json(실패 기록)은 FAQ가 아니고,
    _failures.json이 없는 라운드 파일은 중간 저장본(생성 중이거나 끊긴 라운드)이라 건너뛴다."""
    items = []
    paths = [p for p in INPUT_DIR.glob(f"petmint_{species}_faq_round*.json") if re.fullmatch(rf"petmint_{species}_faq_round\d+\.json", p.name)]
    for path in sorted(paths, key=lambda p: int(re.search(r"round(\d+)", p.name).group(1))):
        r = int(re.search(r"round(\d+)", path.name).group(1))
        if not path.with_name(path.stem + "_failures.json").exists():
            print(f"  {path.name}: 완료 표시(_failures.json) 없음 — 끝나지 않은 생성 라운드라 건너뜀")
            continue
        for i, f in enumerate(json.load(open(path, encoding="utf-8"))):
            items.append({**f, "round": r, "row": i, "checks": []})
    return items

faqs = {sp: load_rounds(sp) for sp in SPECIES}
for sp in SPECIES:
    print(sp, len(faqs[sp]), "QA", dict(Counter(f["round"] for f in faqs[sp])))

강아지 7097 QA {1: 808, 2: 810, 3: 804, 4: 803, 5: 828, 6: 807, 7: 812, 8: 474, 9: 481, 10: 470}
고양이 6663 QA {1: 716, 2: 731, 3: 709, 4: 729, 5: 732, 6: 723, 7: 735, 8: 718, 9: 436, 10: 434}


In [3]:
# ===== 3: 단위 복원 + 결정적 검사 =====
# 참조는 단위마다 한 번만 조회한다(같은 단위 QA가 라운드 수 × 2~3개).
ref_cache = {}
def unit_and_refs(sp: str, uid: str):
    u = units[sp].get(uid)
    if u is None:
        return None, None
    if (sp, uid) not in ref_cache:
        ref_cache[(sp, uid)] = builder.resolve_references(u, sp, source_chunks=u["records"])
    return u, ref_cache[(sp, uid)]

missing = []
for sp in SPECIES:
    for f in faqs[sp]:
        u, refs = unit_and_refs(sp, f["unit_id"])
        if u is None and f["unit_id"] in excluded_units[sp]:      # faq_units.is_intro_only — 원문에 정의가 없어 지어낸 QA
            f["checks"].append({"check": "unit_excluded", "action": "drop", "detail": "도입 문장만 있는 단위(FAQ 제외)"})
            continue
        if u is None:                          # 05 생성 뒤 01~03이 바뀌어 단위가 달라진 경우 — 검사할 근거가 없으므로 버린다
            missing.append((sp, f["unit_id"]))
            f["checks"].append({"check": "unit_missing", "action": "drop", "detail": "현재 데이터에 없는 단위"})
            continue
        body = faq_checks.answer_body(f["answer"])
        used = rag_context.used_references(body, refs + u.get("merged_refs", []), u["content"], unit_article=u.get("article_num"))
        suffix = faq_units.build_citation_suffix(u, used)
        f["checks"] += faq_checks.run_checks(f, rag_context.display_text(u), refs,
                                             [r["paragraph_num"] for r in u["records"]], suffix)
        if any(c["check"] == "citation_suffix" for c in f["checks"]):    # 수정: 근거 표기를 다시 조립
            f["answer"] = f"{body} {suffix}"
            f["references"] = [r["citation_label"] for r in used]
print("현재 데이터에 없는 단위:", len(set(missing)), "| 제외 단위 QA:", sum(c["check"] == "unit_excluded" for sp in SPECIES for f in faqs[sp] for c in f["checks"]))
for sp in SPECIES:
    print(sp, dict(Counter(c["check"] for f in faqs[sp] for c in f["checks"])))

현재 데이터에 없는 단위: 0 | 제외 단위 QA: 45
강아지 {'number_grounding': 211, 'needs_review': 403, 'article_ref': 36, 'unit_excluded': 21, 'care_type': 1, 'limit_value': 1}
고양이 {'needs_review': 429, 'number_grounding': 172, 'unit_excluded': 24, 'article_ref': 13, 'limit_value': 2, 'care_type': 1, 'grade_confusion': 1}


In [4]:
# ===== 4: LLM 심판 (gemma4:12b, Ollama) — 결정적 검사에서 버린 QA만 건너뛴다 =====
# 중복보다 먼저 판정한다: 중복을 먼저 지우면 대표 QA가 심판에 걸렸을 때 판정받지 않은 중복본까지 사라진다(첫 전량 실행에서 721건).
# 중단 후 재개: 노트북을 처음부터 다시 실행하면 된다(셀 1~3은 수 분, 이 셀은 OUTPUT_DIR/judge_cache.json에 있는 판정을 건너뜀)
# 측정: 검수 3회차 시험 재현 88%·OK 오탐 20%·남는 오류율 25%→5%, 전량 실행 약 1초/건 (faq_judge 주석 참고)
import requests
assert faq_judge.JUDGE_MODEL in {m["name"] for m in requests.get(f"{faq_judge.OLLAMA_URL}/api/tags").json()["models"]}, \
    f"{faq_judge.JUDGE_MODEL} 없음 — docker exec ollama ollama pull {faq_judge.JUDGE_MODEL}"

import hashlib
SAVE_EVERY = 200          # 새 판정 200건(약 4분)마다 캐시 저장 — 중단돼도 다시 실행하면 저장된 판정은 건너뛴다

def judge_key(sp, f):
    """질문·답변 해시를 넣어, 05 라운드를 다시 생성해도 옛 판정이 새 QA에 붙지 않게 한다"""
    h = hashlib.md5((f["question"] + "|" + faq_checks.answer_body(f["answer"])).encode()).hexdigest()[:12]
    return f"{sp}|{f['round']}|{f['row']}|{h}"

cache_path = OUTPUT_DIR / "judge_cache.json"
judge_cache = json.load(open(cache_path, encoding="utf-8")) if cache_path.exists() else {}
start, n_new = time.time(), 0
for sp in SPECIES:
    for f in faqs[sp]:
        if faq_checks.decision(f["checks"]) == "drop":
            continue
        k = judge_key(sp, f)
        if k not in judge_cache:
            u, refs = unit_and_refs(sp, f["unit_id"])
            prompt = faq_judge.build_judge_prompt(f, rag_context.display_text(u), refs, faq_checks.answer_body(f["answer"]))
            res = faq_judge.judge(prompt)
            if res["problem"] is None:                         # 파싱 실패 한 번 재시도
                res = faq_judge.judge(prompt)
            judge_cache[k] = res
            n_new += 1
            if n_new % SAVE_EVERY == 0:
                json.dump(judge_cache, open(cache_path, "w", encoding="utf-8"), ensure_ascii=False)
                print(f"  {n_new}건 판정, {(time.time()-start)/60:.1f}분")
        res = judge_cache[k]
        if res["problem"]:
            f["checks"].append({"check": "judge", "action": "drop", "detail": f"{res['types']} {res['reason']}"})
        elif res["problem"] is None:
            f["checks"].append({"check": "judge_fail", "action": "flag", "detail": "심판 출력 파싱 실패"})
        if res.get("truncated"):
            f["checks"].append({"check": "judge_truncated", "action": "flag", "detail": f"입력 {res['prompt_tokens']}토큰 — 잘렸을 수 있음"})
json.dump(judge_cache, open(cache_path, "w", encoding="utf-8"), ensure_ascii=False)
print(f"새로 판정 {n_new}건, {(time.time()-start)/60:.1f}분")
# 중복 셀이 임베딩 모델을 올리기 전에 심판 모델을 GPU에서 내린다
requests.post(f"{faq_judge.OLLAMA_URL}/api/generate", json={"model": faq_judge.JUDGE_MODEL, "keep_alive": 0}, timeout=60)

  200건 판정, 3.6분
  400건 판정, 7.3분
  600건 판정, 10.8분
  800건 판정, 14.4분
  1000건 판정, 17.9분
  1200건 판정, 21.4분
  1400건 판정, 25.0분
  1600건 판정, 28.6분
  1800건 판정, 32.1분
  2000건 판정, 35.7분
  2200건 판정, 39.3분
새로 판정 2295건, 41.0분


<Response [200]>

In [5]:
# ===== 5: 중복 — 심판까지 통과한 QA끼리, 같은 단위 묶음(종·특약·조·등급) 안에서 질문이 같거나 거의 같으면 뒤의 것을 버림 =====
# 이어서 단위당 상한: 남은 QA가 max(5, 원문 길이/40)을 넘으면 큰 군집(바꿔 말한 질문)부터 깎는다 — faq_checks.find_over_cap 주석 참고.
# 특약을 넘나드는 유사도는 중복 신호가 아니다(통원Ⅱ/입원Ⅲ의 같은 질문은 둘 다 필요) — faq_checks 주석 참고.
from sentence_transformers import SentenceTransformer
import torch
embedder = SentenceTransformer("nlpai-lab/KURE-v1", device="cuda")      # 02·04와 같은 임베딩 모델
for sp in SPECIES:
    alive = [f for f in faqs[sp] if faq_checks.decision(f["checks"]) != "drop"]
    emb = embedder.encode([f["question"] for f in alive], normalize_embeddings=True, batch_size=64, show_progress_bar=False)
    for i, res in faq_checks.find_duplicates(alive, emb).items():
        alive[i]["checks"].append(res)
    print(sp, "중복", sum(any(c["check"] == "duplicate" for c in f["checks"]) for f in faqs[sp]))
    keep = [i for i, f in enumerate(alive) if faq_checks.decision(f["checks"]) != "drop"]
    content_len = {uid: len(u["content"]) for uid, u in units[sp].items()}
    for j, res in faq_checks.find_over_cap([alive[i] for i in keep], emb[keep], content_len).items():
        alive[keep[j]]["checks"].append(res)
    print(sp, "단위 상한", sum(any(c["check"] == "unit_cap" for c in f["checks"]) for f in faqs[sp]))
del embedder
torch.cuda.empty_cache()

/home/kopo17/llm_env/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 391/391 [00:00<00:00, 15704.04it/s]


강아지 중복 996
강아지 단위 상한 333
고양이 중복 1028
고양이 단위 상한 406


In [6]:
# ===== 6: 결정 + 내보내기 =====
# final: drop이 아닌 QA(표시가 있으면 flags로 남김) / review_queue: 표시가 붙은 QA / all: 검사 기록 전체
for sp in SPECIES:
    for f in faqs[sp]:
        f["decision"] = faq_checks.decision(f["checks"])
    final = [{k: v for k, v in f.items() if k not in ("checks", "decision")} | {"flags": [c["check"] for c in f["checks"] if c["action"] == "flag"]}
             for f in faqs[sp] if f["decision"] != "drop"]
    queue = [f for f in faqs[sp] if f["decision"] == "flag"]
    json.dump(final, open(OUTPUT_DIR / f"petmint_{sp}_faq_final.json", "w", encoding="utf-8"), ensure_ascii=False, indent=2)
    json.dump(queue, open(OUTPUT_DIR / f"petmint_{sp}_review_queue.json", "w", encoding="utf-8"), ensure_ascii=False, indent=2)
    json.dump(faqs[sp], open(OUTPUT_DIR / f"petmint_{sp}_faq_checked_all.json", "w", encoding="utf-8"), ensure_ascii=False, indent=2)
    dc = Counter(f["decision"] for f in faqs[sp])
    drop_by = Counter(c["check"] for f in faqs[sp] for c in f["checks"] if c["action"] == "drop")
    print(f"{sp}: 입력 {len(faqs[sp])} → 최종 {len(final)} ({len(final)/max(len(faqs[sp]),1):.0%}) | 결정 {dict(dc)} | 버림 사유 {dict(drop_by)}")

강아지: 입력 7097 → 최종 2779 (39%) | 결정 {'drop': 4318, 'pass': 2623, 'flag': 156} | 버림 사유 {'judge': 2968, 'unit_cap': 333, 'duplicate': 996, 'unit_excluded': 21}
고양이: 입력 6663 → 최종 2564 (38%) | 결정 {'pass': 2390, 'drop': 4099, 'flag': 174} | 버림 사유 {'judge': 2640, 'unit_excluded': 24, 'unit_cap': 406, 'duplicate': 1028, 'grade_confusion': 1}


In [7]:
# ===== 7: 단위별 잔여 QA — 오탐이 한 주제에 몰리면 챗봇이 그 주제를 통째로 모르게 된다 =====
# 0이 된 단위는 반드시 보고한다. 생성은 됐는데 전부 버려진 단위와, 05에서 아예 생성되지 않은 단위(파싱 실패 등)를 나눠 본다.
# 잔여율은 단위 상한(unit_cap)으로 깎은 QA를 분모에서 뺀다 — 상한은 일부러 줄인 것이라 오탐 쏠림 신호가 아니다.
LOW_RATIO = 0.25
SHOW = 30
def unit_label(c):
    art = f" 제{c['article_num']}조({c['article_title']})" if c["article_num"] is not None else ""
    return f"{c['section_name'] or c['section']}{art} {c['coverage_type'] or ''}".strip()
coverage = []
for sp in SPECIES:
    before, after, capped = Counter(), Counter(), Counter()
    for f in faqs[sp]:
        before[f["unit_id"]] += 1
        capped[f["unit_id"]] += any(c["check"] == "unit_cap" for c in f["checks"])
        after[f["unit_id"]] += f["decision"] != "drop"
    for uid, u in units[sp].items():
        b, a = before.get(uid, 0), after.get(uid, 0)
        reasons = Counter(c["check"] for f in faqs[sp] if f["unit_id"] == uid for c in f["checks"] if c["action"] == "drop") if b and not a else {}
        coverage.append({"species": sp, "unit_id": uid, "section": u["section"], "section_name": u.get("section_name"),
                         "article_num": u.get("article_num"), "article_title": u.get("article_title"),
                         "coverage_type": u.get("coverage_type"), "generated": b, "kept": a, "capped": capped.get(uid, 0),
                         "drop_reasons": dict(reasons)})
json.dump(coverage, open(OUTPUT_DIR / "unit_coverage.json", "w", encoding="utf-8"), ensure_ascii=False, indent=2)
for sp in SPECIES:
    cv = [c for c in coverage if c["species"] == sp]
    zero_gen = [c for c in cv if c["generated"] == 0]
    zero_kept = [c for c in cv if c["generated"] > 0 and c["kept"] == 0]
    low = [c for c in cv if c["generated"] > 0 and 0 < c["kept"] / (c["generated"] - c["capped"]) < LOW_RATIO]
    print(f"\n[{sp}] 단위 {len(cv)} | 생성 안 됨 {len(zero_gen)} | 전부 버려짐 {len(zero_kept)} | 잔여 {LOW_RATIO:.0%} 미만 {len(low)}")
    for c in zero_kept[:SHOW]:
        print(f"  전부 버려짐: {unit_label(c)} — {c['generated']}개 중 0, 사유 {c['drop_reasons']}")
    for c in zero_gen[:SHOW]:
        print(f"  생성 안 됨: {unit_label(c)}")
    if max(len(zero_kept), len(zero_gen)) > SHOW:
        print(f"  … 전체 목록은 unit_coverage.json")
# 섹션별 잔여율 — 특정 특약·주제에 버림이 몰렸는지
by_sec = defaultdict(lambda: [0, 0])
for c in coverage:
    by_sec[(c["species"], c["section_name"] or c["section"])][0] += c["generated"] - c["capped"]
    by_sec[(c["species"], c["section_name"] or c["section"])][1] += c["kept"]
print("\n잔여율 낮은 섹션 하위 10:")
for (sp, name), (g, k) in sorted(((key, v) for key, v in by_sec.items() if v[0]), key=lambda kv: kv[1][1] / kv[1][0])[:10]:
    print(f"  {sp} {name}: {k}/{g} ({k/max(g,1):.0%})")


[강아지] 단위 301 | 생성 안 됨 0 | 전부 버려짐 0 | 잔여 25% 미만 36

[고양이] 단위 274 | 생성 안 됨 0 | 전부 버려짐 0 | 잔여 25% 미만 33

잔여율 낮은 섹션 하위 10:
  고양이 【별표4】 민사소송 인지액 /【별표5】 송달료: 3/15 (20%)
  강아지 보험금 지급 절차: 9/40 (22%)
  강아지 【별표4】 민사소송 인지액 /【별표5】 송달료: 3/12 (25%)
  고양이 가입자 유의사항: 5/20 (25%)
  고양이 지정대리청구서비스 특별약관: 50/172 (29%)
  강아지 보험용어 해설: 6/20 (30%)
  고양이 보험료 자동납입 특별약관: 19/62 (31%)
  강아지 【별표3】 소송목적의 값에 따른 변호사비용: 5/16 (31%)
  고양이 【별표3】 소송목적의 값에 따른 변호사비용: 5/16 (31%)
  고양이 특정질병 분류표(반려묘): 32/100 (32%)
